In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd

from automed import AutoMed, Dataset

[11:26:28] cuDF not found: falling back to standalone pandas.

In [2]:
titanic = pd.read_csv('../perf_logger/tests_data/titanic.csv', delimiter=';')

autom = AutoMed()

In [3]:
print(autom.debug_load())
print(autom.json_pipeline())

None
{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'RandomSplit', 'name': 'Split date to train and test set', 'description': 'Step description...', 'configuration': {'ratio': {'description': 'Split ratio', 'default': 0.2, 'value': 0.2}, 'random_state': {'description': 'Random state', 'default': 42, 'value': 42}}, 'children': []}, {'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActOnehot', 'name': 'One hot encoding categorical features', 'description': 'Step description...', 'configuration': {}, 'children': []}, {'step': 'ActDropNumericalColumn', 'name': 'Drop Numerical Column', 'description': 'Step description...', 'configuration': {'empty_threshold': {'description': 'Column with more or equal proportion of empty row will dropped. 1 will drop all columns', 'default': 0.5, 'value': 0.5}}, 'children': []}, {'step': 'ActDropDateColum

In [4]:
pipeline = {
    'step': 'MetaOrderedStep',
    'children': [{
        'step': 'RandomSplit',
        'configuration': {
            'ratio': {
                'value': 0.3
            }
        }},
        {
            'step': 'MetaStep',
            'tag': 'cleaning'
        },
        {
            'step': 'MetaStep',
            'tag': 'metric'
        },
        {
            'step': 'MetaExplorerStep',
            'tag': 'learning'
        }]
}

autom.load_pipeline(pipeline)
print(autom.json_pipeline())


{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'RandomSplit', 'name': 'Split date to train and test set', 'description': 'Step description...', 'configuration': {'ratio': {'description': 'Split ratio', 'default': 0.2, 'value': 0.3}, 'random_state': {'description': 'Random state', 'default': 42, 'value': 42}}, 'children': []}, {'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActOnehot', 'name': 'One hot encoding categorical features', 'description': 'Step description...', 'configuration': {}, 'children': []}, {'step': 'ActDropNumericalColumn', 'name': 'Drop Numerical Column', 'description': 'Step description...', 'configuration': {'empty_threshold': {'description': 'Column with more or equal proportion of empty row will dropped. 1 will drop all columns', 'default': 0.5, 'value': 0.5}}, 'children': []}, {'step': 'ActDropDateColumn', '

In [5]:
print(titanic[['label']])
results = autom.fit(
    X=titanic.drop('label', axis=1),
    Y=titanic[['label']],
    callback=lambda step: print(step, id(step), step.parents_steps))
[ (r.model.ml_model, r.evaluate()) for r in results if r.model.ml_model is not None ]

     label
0        0
1        1
2        1
3        1
4        0
..     ...
886      0
887      1
888      0
889      1
890      0

[891 rows x 1 columns]


Output()

[11:26:29] running step: MetaOrderedStep (steps=MetaStep,RandomSplit,MetaExplorerStep)

Split date to train and test set 2029870395984 [2029845856656]

           running step: MetaStep                                                                                  
           (steps=ActDropDateColumn,ActDropTextualColumn,ActOnehot,ActTfIdf,ActMeanColumn,ActDropNumericalColumn,Ac
           tSplitDate)

...

Learn : Logistic regression 2029870434960 [2029870432400, 2029845856656]

Learn : XGBoost 2029870434128 [2029870432400, 2029845856656]

MetaExplorerStep 2029870432400 [2029845856656]

MetaStep 2029845856656 []

[(LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42),
  0.8139094508521261),
 (RandomForestClassifier(max_depth=15, random_state=42), 0.7953463017157285),
 (GradientBoostingClassifier(learning_rate=1.0, max_depth=15, random_state=42),
  0.6954438514948069),
 (KNeighborsClassifier(), 0.5966029723991507),
 (GaussianNB(), 0.5938486256957595),
 (SVC(random_state=42), 0.5336259826705687)]

In [ ]:
print(results[0].model.sklearn_model)
print(results[0].model.stack)

pm = results[0].model.pickle()
[ len(r.model.pickle()) for r in results ]

In [ ]:
import pickle

o = 200 # offset
n = 68  # # of samples
labels = titanic.iloc[o:(o+n)]['label']
predict_df = titanic.iloc[o:(o+n)].drop('label', axis=1).copy()

# labels = labels.reset_index()
predict_df.reset_index(inplace=True, drop=True)

m = pickle.loads(pm)
sum([ r == labels[o+i] for i, r in enumerate(m.run(predict_df)) ]) / n

In [3]:
final_boss_dataset = Dataset(titanic.copy())
final_boss_dataset.set_label('label')

final_boss_automed = AutoMed(final_boss_dataset, max_workers=2)
final_boss_automed.debug_load()
final_boss_results = final_boss_automed.run()

Output()

[16:04:29] running step: MetaOrderedStep (steps=RandomSplit,MetaStep,MetaExplorerStep)

           running step: MetaStep                                                                                  
           (steps=ActSplitDate,ActDropTextualColumn,ActTfIdf,ActOnehot,ActDropNumericalColumn,ActMeanColumn,ActDrop
           DateColumn)

           running step: MetaStep (steps=ActRemoveHighCorrelatedColumn)

[16:04:34] running step: MetaStep (steps=ActRandomOverSampling,ActMinMaxScaler)

...

1 1 2

1 1 2

1 1 2

1 1 2

[16:06:54] finished all generations: ActXGBoost

In [4]:
# sum([ len(r.model.pickle()) for r in final_boss_results ])
[ (r.model.ml_model, r.evaluate()) for r in final_boss_results if r.model.ml_model is not None ]

[(GradientBoostingClassifier(learning_rate=0.9269510175667243, max_depth=19,
                             n_estimators=25, random_state=42),
  0.8204633204633205),
 (GradientBoostingClassifier(learning_rate=0.9894177858428046, max_depth=35,
                             n_estimators=213, random_state=42),
  0.8149292149292149),
 (SVC(class_weight='balanced', random_state=42), 0.8117117117117117),
 (RandomForestClassifier(max_depth=62, n_estimators=499, random_state=42),
  0.808944658944659),
 (LogisticRegression(max_iter=4636, n_jobs=-1, random_state=42),
  0.8041827541827542),
 (LogisticRegression(max_iter=1827, n_jobs=-1, random_state=42),
  0.8041827541827542),
 (LogisticRegression(max_iter=3921, n_jobs=-1, random_state=42),
  0.8041827541827542),
 (LogisticRegression(max_iter=1861, n_jobs=-1, random_state=42),
  0.8041827541827542),
 (LogisticRegression(max_iter=4115, n_jobs=-1, random_state=42),
  0.8041827541827542),
 (LogisticRegression(max_iter=4413, n_jobs=-1, random_state=42),